# Module 6 Homework: Batch Processing with Spark

Based on [Module 6 Homework](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/cohorts/2026/06-batch/homework.md).

This notebook walks through the questions and answers. Install Spark and PySpark per the [module setup](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/06-batch/setup), download the datasets below, then run the cells in order.

**Run this notebook**

1. Activate your environment (e.g. `conda activate dataTalks`), then open this notebook in Jupyter.
2. Download data into `HW6_DIR` (see **Setup**): Yellow November 2025 Parquet and the taxi zone lookup CSV.
3. Run the **Setup** cell first (Spark session and paths).
4. Run each section in order; code cells compute the homework answers when the files are present.

**Downloads** (from the homework page):

```bash
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Place them in the same folder this notebook resolves as `DATA_DIR` (by default `06-batch/HW6/data`), or set paths in the Setup cell.

In [ ]:
# Setup: paths, Spark session, spark.version (run this first)
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

cwd = Path.cwd()
# Resolve HW6 folder: same dir as this notebook, or repo 06-batch/HW6
HW6_DIR = None
for p in (cwd, cwd / "06-batch" / "HW6", cwd.parent, cwd / "HW6"):
    if p and (p / "hw6.ipynb").exists():
        HW6_DIR = p.resolve()
        break
if HW6_DIR is None:
    HW6_DIR = (cwd / "06-batch" / "HW6").resolve()

DATA_DIR = HW6_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

YELLOW_PARQUET = DATA_DIR / "yellow_tripdata_2025-11.parquet"
ZONE_CSV = DATA_DIR / "taxi_zone_lookup.csv"
OUT_PARQUET_DIR = DATA_DIR / "yellow_2025_11_repartitioned"

spark = (
    SparkSession.builder.master("local[*]").appName("hw6").getOrCreate()
)

print("HW6_DIR:", HW6_DIR)
print("YELLOW_PARQUET exists:", YELLOW_PARQUET.exists(), YELLOW_PARQUET)
print("ZONE_CSV exists:", ZONE_CSV.exists(), ZONE_CSV)
print("spark.version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/30 12:16:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


HW6_DIR: /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/06-batch/HW6
YELLOW_PARQUET exists: True /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/06-batch/HW6/data/yellow_tripdata_2025-11.parquet
ZONE_CSV exists: True /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/06-batch/HW6/data/taxi_zone_lookup.csv
spark.version: 4.1.1


### Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local Spark session
- Execute `spark.version`

What is the output?

> Use the [setup guide](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/06-batch/setup/) for installation.

**Answer:** The printed `spark.version` string (for example `3.5.x` depending on your Spark build). Submit the exact value your environment prints after running the Setup cell.

### Question 2: Yellow November 2025

Read the November 2025 Yellow data into a Spark DataFrame. Repartition to 4 partitions and save as Parquet.

What is the average size of the Parquet **files** (files ending in `.parquet`) in MB? Pick the closest option.

- 6MB
- 25MB
- 75MB
- 100MB

In [ ]:
# Part 2: read, repartition(4), write; then average .parquet file size in MB
if not YELLOW_PARQUET.exists():
    raise FileNotFoundError(f"Download yellow parquet to: {YELLOW_PARQUET}")

df_yellow = spark.read.parquet(str(YELLOW_PARQUET))
df_yellow = df_yellow.repartition(4)

OUT_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
df_yellow.write.mode("overwrite").parquet(str(OUT_PARQUET_DIR))

parquet_files = list(OUT_PARQUET_DIR.rglob("*.parquet"))
sizes_mb = [p.stat().st_size / (1024 * 1024) for p in parquet_files]
avg_mb = sum(sizes_mb) / len(sizes_mb) if sizes_mb else 0
print(f"Number of .parquet part files: {len(sizes_mb)}")
print(f"Average size (MB): {avg_mb:.2f}")

Number of .parquet part files: 4
Average size (MB): 24.42


**Answer:** Use the printed `Average size (MB)` and pick the closest option among 6, 25, 75, or 100 MB.

### Question 3: Count records

How many taxi trips started on the **15th of November** (November 15 only)?

- 62,610
- 102,340
- 162,604
- 225,768

In [ ]:
# Part 3: trips with pickup on 2025-11-15 (Yellow uses tpep_pickup_datetime)
df = spark.read.parquet(str(YELLOW_PARQUET))
cnt = (
    df.withColumn("pickup_date", F.to_date(F.col("tpep_pickup_datetime")))
    .filter(F.col("pickup_date") == F.lit("2025-11-15"))
    .count()
)
print(cnt)

162604


**Answer:** Match the printed count to one of the options (the homework expects one specific value — run the cell after placing the Parquet file).

### Question 4: Longest trip

What is the length of the longest trip in the dataset, in **hours**?

- 22.7
- 58.2
- 90.6
- 134.5

In [ ]:
# Part 4: max (dropoff - pickup) in hours
df = spark.read.parquet(str(YELLOW_PARQUET))
max_row = df.select(
    (
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime"))
        / F.lit(3600.0)
    ).alias("trip_hours")
).agg(F.max("trip_hours").alias("max_hours"))

max_row.show()
print("max_hours:", max_row.collect()[0]["max_hours"])

+-----------------+
|        max_hours|
+-----------------+
|90.64666666666666|
+-----------------+

max_hours: 90.64666666666666


**Answer:** Compare `max_hours` to the choices (one of 22.7, 58.2, 90.6, 134.5).

### Question 5: User Interface

Spark's UI (application dashboard) runs on which local port by default?

- 80
- 443
- 4040
- 8080

**Answer:** **4040** — the Spark UI for the driver application listens on port 4040 by default (while the session is running).

### Question 6: Least frequent pickup location zone

Load the zone lookup into a temp view, join to Yellow November 2025 data, and find the **name** of the **least** frequent pickup zone.

- Governor's Island/Ellis Island/Liberty Island
- Arden Heights
- Rikers Island
- Jamaica Bay

If multiple zones tie for least frequency, any of those names is acceptable.

In [ ]:
# Part 6: zones + yellow; least frequent PULocationID by zone name
if not ZONE_CSV.exists():
    raise FileNotFoundError(f"Download zone CSV to: {ZONE_CSV}")

df = spark.read.parquet(str(YELLOW_PARQUET))
zones = spark.read.option("header", True).option("inferSchema", True).csv(str(ZONE_CSV))

zones.createOrReplaceTempView("zones")
df.createOrReplaceTempView("yellow")

least = spark.sql(
    """
    SELECT z.Zone, COUNT(*) AS c
    FROM yellow y
    JOIN zones z ON y.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY c ASC
    LIMIT 5
    """
)
least.show(truncate=False)

+---------------------------------------------+---+
|Zone                                         |c  |
+---------------------------------------------+---+
|Governor's Island/Ellis Island/Liberty Island|1  |
|Eltingville/Annadale/Prince's Bay            |1  |
|Arden Heights                                |1  |
|Port Richmond                                |3  |
|Rikers Island                                |4  |
+---------------------------------------------+---+



**Answer:** The `Zone` in the first row after `ORDER BY c ASC` (least pickups). Match it to one of the four choices above.

---

## Submitting the solutions

Form: https://courses.datatalks.club/de-zoomcamp-2026/homework/hw6

Deadline: see the [course homework page](https://courses.datatalks.club/de-zoomcamp-2026/).